In [4]:
import subprocess, sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try: __import__(name)
    except ImportError:
        for extra in [[], ["--break-system-packages"]]:
            if subprocess.call([sys.executable, "-m", "pip", "install", pkg, "-q"] + extra,
                               stderr=subprocess.DEVNULL) == 0:
                break

_ensure("reportlab"); _ensure("matplotlib"); _ensure("fonttools")

import pandas as pd
import numpy as np
import requests, io, logging, os, tempfile, platform as _platform
from pathlib import Path
from datetime import datetime

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                Table, TableStyle, Image,
                                HRFlowable, KeepTogether)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# ══════════════════════════════════════════════════════════════════════════════
PLATFORMS = [
    {
        "name":       "Afun_mx",
        "excel_dir":  r"D:\Afun_mx自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "Soluno",
        "excel_dir":  r"D:\soluno自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "sol777",
        "excel_dir":  r"D:\sol777自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "Lucro",
        "excel_dir":  r"D:\Lucro自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "3bet",
        "excel_dir":  r"D:\3bet自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "A7X",
        "excel_dir":  r"D:\A7X自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "Wey7",
        "excel_dir":  r"D:\wey7自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "oro7x",
        "excel_dir":  r"D:\oro7x自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "Letjoy",
        "excel_dir":  r"D:\Letjoy自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
    },
    {
        "name":       "G777",
        "dept":       "",
        "channel":    "短信/投放",
        "excel_dir":  r"D:\G777自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
        "n_days":     35,
    },
    {
        "name":       "Luna777",
        "dept":       "",
        "channel":    "短信/投放",
        "excel_dir":  r"D:\Luna777自动播报",
        "excel_glob": "日报-大盘日报_*.xlsx",
        "output_dir": r"D:\自动播报报告",
        "bot_token":  "8782270942:AAFng-sm_nckrJpdgrinkcx-77iYOYuDy2k",
        "chat_id":    "-5258799427",
        "n_days":     35,
    },
]

# ★ 留存目标
RETENTION_TARGETS = [
    ("2日复充",  "首充2日复充率",  21.0,  1),
    ("3日复充",  "首充3日复充率",  15.0,  2),
    ("7日复充",  "首充7日复充率",  11.0,  6),
    ("14日复充", "首充14日复充率",  8.0, 13),
    ("30日复充", "首充30日复充率",  6.0, 29),
]

# ══════════════════════════════════════════════════════════════════════════════
# 字段名常量
# ══════════════════════════════════════════════════════════════════════════════
CAC_COL      = "一级首充获客成本"        # 一级首充获客成本列名
FISSION_COL  = "非一级充值人数/充值人数"  # Excel 直接提供的裂变率字段名

# ══════════════════════════════════════════════════════════════════════════════
# 字体初始化
# ══════════════════════════════════════════════════════════════════════════════
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s",
                    datefmt="%H:%M:%S", handlers=[logging.StreamHandler(sys.stdout)])
log = logging.getLogger(__name__)


def _init_font() -> str:
    from fontTools.ttLib import TTFont as FTF, TTCollection
    cache = os.path.join(tempfile.gettempdir(), "cjk_rl_font.ttf")
    if os.path.exists(cache):
        try:
            if "glyf" in FTF(cache): return cache
        except Exception: pass
    sys_name = _platform.system()
    candidates = {
        "Windows": [os.path.join(os.environ.get("WINDIR", r"C:\Windows"), "Fonts", n)
                    for n in ["msyh.ttc", "simhei.ttf", "simsun.ttc", "msyhbd.ttc"]],
        "Darwin":  ["/System/Library/Fonts/PingFang.ttc",
                    "/Library/Fonts/Arial Unicode.ttf"],
    }.get(sys_name, [
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    ])
    def _try(path):
        try:
            ext = os.path.splitext(path)[1].lower()
            if ext == ".ttf":
                f = FTF(path)
                if "glyf" in f: return path
            else:
                for f in TTCollection(path):
                    if "glyf" in f: f.save(cache); return cache
        except Exception: pass
    for p in candidates:
        if os.path.exists(p):
            r = _try(p)
            if r: return r
    import urllib.request
    log.info("下载备用字体 WQY MicroHei …")
    wqy = os.path.join(tempfile.gettempdir(), "wqy-microhei.ttc")
    urllib.request.urlretrieve(
        "https://github.com/anthonyfok/fonts-wqy-microhei/raw/master/wqy-microhei.ttc", wqy)
    r = _try(wqy)
    if r: return r
    raise RuntimeError("无法找到可用的 TrueType 中文字体。")


_FONT_PATH = _init_font()
pdfmetrics.registerFont(TTFont("NotoSC",   _FONT_PATH))
pdfmetrics.registerFont(TTFont("NotoSC-B", _FONT_PATH))
_FP = fm.FontProperties(fname=_FONT_PATH)
plt.rcParams["axes.unicode_minus"] = False

# ══════════════════════════════════════════════════════════════════════════════
# 调色板
# ══════════════════════════════════════════════════════════════════════════════
C_GOLD   = "#B5922A"; C_DARK  = "#2C2416"; C_RED    = "#C04A3A"
C_GREEN  = "#3D7A5E"; C_BLUE  = "#4A7AB5"; C_MUTED  = "#8A7A68"
C_ORANGE = "#D4743A"; C_GRAY  = "#AAAAAA"

# ══════════════════════════════════════════════════════════════════════════════
# 1. 数据加载
# ══════════════════════════════════════════════════════════════════════════════

def find_excel(cfg) -> Path:
    files = sorted(Path(cfg["excel_dir"]).glob(cfg["excel_glob"]),
                   key=lambda f: f.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError(f"[{cfg['name']}] 未找到 {cfg['excel_glob']}")
    log.info(f"[{cfg['name']}] 读取：{files[0].name}")
    return files[0]


def clean_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False),
        errors="coerce")


def load_df(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path)
    for col in df.columns[1:]:
        df[col] = clean_num(df[col])

    df = df.sort_values("日期", ascending=True).reset_index(drop=True)
    df.rename(columns={"游戏盈亏投注比": "游戏盈利率"}, inplace=True)

    log.info(f"[load_df] Excel 列名列表：{df.columns.tolist()}")

    if CAC_COL not in df.columns:
        log.warning(f"[load_df] 未找到列「{CAC_COL}」，获客成本将显示 N/A。")
    if FISSION_COL not in df.columns:
        log.warning(f"[load_df] 未找到列「{FISSION_COL}」，裂变率将显示 N/A。")

    df["日期标签"] = df["日期"].astype(str).apply(lambda s: f"{s[4:6]}/{s[6:8]}")

    df["老用户ARPPU"] = (df["老用户充值金额"] /
        (df["充值人数"] - df["首充人数"]).replace(0, np.nan)).round(3)
    df["累计毛利ROI"] = (df["历史累计毛利"] /
        df["历史累计真实消耗"].replace(0, np.nan) * 100).round(2)
    df["日充提差ROI"] = (
        df["充提差"] / df["真实消耗"].replace(0, np.nan) * 100
    ).round(2)


    if FISSION_COL in df.columns:
        raw = df[FISSION_COL].copy()
        # 自动判断：若所有非空值均 <= 1，则视为小数，乘 100 转换
        non_null = raw.dropna()
        if len(non_null) > 0 and non_null.abs().max() <= 1:
            raw = (raw * 100).round(2)
        df["裂变率"] = raw
    else:
        df["裂变率"] = np.nan

    # 充值=0 时置 NaN
    zero_dep = df["充值金额"].fillna(0) <= 0
    null_cols = [
        "充提差", "充提差比", "充值人数", "人均充值金额", "首充人数",
        "老用户充值金额", "投注金额", "新充付费率", "老充付费率",
        "首充转化率", "总赠送充值比", "游戏盈利率",
        "首充2日活跃留存率", "首充2日复充率", "首充3日活跃留存率",
        "首充7日活跃留存率", "首充3日复充率", "首充7日复充率",
        "首充14日复充率", "首充30日复充率",
        "首充LT日", "首充LTV14日", "首充LTV30日",
        "裂变率",
    ]
    for c in null_cols:
        if c in df.columns:
            df.loc[zero_dep, c] = np.nan

    # 最新一天真实消耗、获客成本置 NaN（当日未完整）
    df.loc[df.index[-1], "真实消耗"] = np.nan
    if CAC_COL in df.columns:
        df.loc[df.index[-1], CAC_COL] = np.nan

    zero_cost = df["真实消耗"].fillna(0) <= 0
    if CAC_COL in df.columns:
        df.loc[zero_cost, CAC_COL] = np.nan

    # 首充 < 50 时留存置 NaN
    small = df["首充人数"].fillna(0) < 50
    for c in ["首充2日活跃留存率", "首充2日复充率", "首充3日活跃留存率",
              "首充7日活跃留存率", "首充3日复充率", "首充7日复充率",
              "首充14日复充率", "首充30日复充率"]:
        if c in df.columns:
            df.loc[small, c] = np.nan
            df[c] = df[c].clip(0, 100)
    return df


# ══════════════════════════════════════════════════════════════════════════════

def valid_df(df: pd.DataFrame) -> pd.DataFrame:
    """返回充值金额 > 0 的行，重置索引，用于绘图。
    这样图表 x 轴只显示真实有业务数据的日期，消除空白占位。"""
    mask = df["充值金额"].fillna(0) > 0
    return df[mask].reset_index(drop=True)


# ══════════════════════════════════════════════════════════════════════════════
# 2. 留存工具
# ══════════════════════════════════════════════════════════════════════════════

def get_complete_series(df, col, lag):
    """窗口已关闭且值>0的全量数据。
    ★ ：接受原始 df（含完整日历），cutoff 基于实际行数。
    """
    n = len(df)
    cutoff = max(0, n - lag)
    xs, vs, ds = [], [], []
    # 用有效行的连续索引映射（仅对有效行）
    valid_idx = 0
    for i in range(n):
        is_valid = df["充值金额"].fillna(0).iloc[i] > 0
        v = df[col].iloc[i] if col in df.columns else np.nan
        if i < cutoff and is_valid and pd.notna(v) and v > 0:
            xs.append(valid_idx)
            vs.append(float(v))
            ds.append(df["日期标签"].iloc[i])
        if is_valid:
            valid_idx += 1
    return xs, vs, ds


def weighted_avg_7d(df, col, lag):
    """近7天完整数据加权均值，基于原始 df。"""
    n = len(df)
    lci = n - lag - 1
    if lci < 0:
        return np.nan, "N/A", "N/A", 0
    s0 = max(0, lci - 6); s1 = lci
    sub_r = df[col].iloc[s0:s1 + 1]
    sub_f = df["首充人数"].iloc[s0:s1 + 1]
    sub_d = df["日期标签"].iloc[s0:s1 + 1]
    mask = (sub_r > 0) & sub_r.notna() & sub_f.notna() & (sub_f > 0)
    r, f, d = sub_r[mask], sub_f[mask], sub_d[mask]
    if f.sum() == 0:
        return np.nan, "N/A", "N/A", 0
    return (float((r * f).sum() / f.sum()),
            d.iloc[0] if len(d) else "N/A",
            d.iloc[-1] if len(d) else "N/A",
            int(mask.sum()))


# ══════════════════════════════════════════════════════════════════════════════

def _fig(w=16, h=5):
    fig, ax = plt.subplots(figsize=(w, h), facecolor="white")
    ax.set_facecolor("white")
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#D8CEBD")
    ax.tick_params(colors=C_MUTED, labelsize=7)
    return fig, ax


def _label_bars(ax, bars, fmt="{:.0f}", size=7, color=C_DARK, offset=3):
    for bar in bars:
        h = bar.get_height()
        if h and not np.isnan(h) and abs(h) > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + offset,
                    fmt.format(h), ha="center", va="bottom",
                    fontsize=size, color=color, fontproperties=_FP,
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.8))


def _label_line(ax, xs, ys, fmt="{:.1f}", size=7, color=C_DARK, offset=8, skip=4):
    for i, (x, y) in enumerate(zip(xs, ys)):
        if y is None or (isinstance(y, float) and np.isnan(y)): continue
        if i % skip != 0: continue
        ax.annotate(fmt.format(y), (x, y), textcoords="offset points", xytext=(0, offset),
                    ha="center", va="bottom", fontsize=size, color=color,
                    fontproperties=_FP,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec=color, alpha=0.88, lw=0.5))


def _to_img(fig, dpi=150) -> io.BytesIO:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight",
                facecolor="white", edgecolor="none")
    plt.close(fig); buf.seek(0); return buf


def _x(vdf):  return list(range(len(vdf)))
def _lb(vdf): return vdf["日期标签"].tolist()


def _xticks(ax, vdf):
    """★ ：基于过滤后的有效数据，自适应步长，最多 20 个标签。"""
    xs = _x(vdf); lb = _lb(vdf)
    step = max(1, len(xs) // 20)
    ax.set_xticks(xs[::step])
    ax.set_xticklabels(lb[::step], fontproperties=_FP, fontsize=7)


# ══════════════════════════════════════════════════════════════════════════════

def chart_dep_cost(vdf):
    """日充值 vs 日真实消耗（均基于有效日期）"""
    fig, ax = _fig(16, 4.5); ax2 = ax.twinx(); xs = _x(vdf)
    bars = ax.bar(xs, vdf["充值金额"].tolist(), color=C_GOLD, alpha=0.7, width=0.6)
    _label_bars(ax, bars, fmt="{:,.0f}", size=7, offset=5)
    ax2.plot(xs, vdf["真实消耗"].tolist(), color=C_RED, lw=1.8, marker="o", ms=3)
    _label_line(ax2, xs, vdf["真实消耗"].tolist(), fmt="{:,.0f}", size=7, color=C_RED, skip=4)
    _xticks(ax, vdf)
    ax.set_ylabel("充值金额", fontproperties=_FP, fontsize=8)
    ax2.set_ylabel("真实消耗", fontproperties=_FP, fontsize=8)
    ax2.spines[["top"]].set_visible(False)
    ax.set_title("日充值金额 vs 日真实消耗（消耗已排除最新一天未完整数据）",
                 fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.legend([Patch(color=C_GOLD, alpha=0.7), Line2D([0], [0], color=C_RED, lw=2, marker='o')],
              ["充值金额", "真实消耗"], prop=_FP, fontsize=8, loc="upper left", framealpha=0.7)
    fig.tight_layout(); return _to_img(fig)


def chart_diff(vdf):
    fig, ax = _fig(16, 4.5); ax2 = ax.twinx(); xs = _x(vdf)
    bars = []
    for x, v in zip(xs, vdf["充提差"].tolist()):
        if pd.notna(v):
            b = ax.bar(x, v, color=C_GREEN if v >= 0 else C_RED, alpha=0.7, width=0.6)
            bars.append(b[0])
    _label_bars(ax, bars, fmt="{:,.0f}", size=7)
    ax2.plot(xs, vdf["充提差比"].tolist(), color=C_GOLD, lw=1.8, marker="D", ms=3)
    _label_line(ax2, xs, vdf["充提差比"].tolist(), fmt="{:.1f}%", size=7, color=C_GOLD, skip=4)
    _xticks(ax, vdf)
    ax.set_ylabel("充提差", fontproperties=_FP, fontsize=8)
    ax2.set_ylabel("充提差比%", fontproperties=_FP, fontsize=8)
    ax2.spines["top"].set_visible(False)
    ax.set_title("充提差 & 充提差比", fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.axhline(0, color="#CCCCCC", lw=0.8, ls="--")
    fig.tight_layout(); return _to_img(fig)


def chart_roi(vdf):
    fig, ax = _fig(16, 5); xs = _x(vdf)
    dr  = vdf["充提差比"].tolist()
    roi = vdf["累计毛利ROI"].tolist()
    ax.fill_between(xs, dr, alpha=0.12, color=C_GOLD)
    ax.plot(xs, dr,  color=C_GOLD,  lw=1.8, marker="o", ms=3, label="当日充提差比%")
    _label_line(ax, xs, dr,  fmt="{:.1f}%", size=7, color=C_GOLD, skip=4)
    ax.plot(xs, roi, color=C_GREEN, lw=2, ls="--", marker="s", ms=3, label="累计毛利ROI%")
    _label_line(ax, xs, roi, fmt="{:.1f}%", size=7, color=C_GREEN, offset=10, skip=4)
    _xticks(ax, vdf)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
    ax.set_title("充提差比 vs 累计毛利ROI（历史累计毛利÷历史累计真实消耗）",
                 fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.legend(prop=_FP, fontsize=8, framealpha=0.7, loc="upper left")
    fig.tight_layout(); return _to_img(fig)


def chart_cum_profit(vdf):
    fig, ax = _fig(16, 5); ax2 = ax.twinx(); xs = _x(vdf)
    bars = ax.bar(xs, vdf["历史累计真实消耗"].tolist(), color=C_MUTED, alpha=0.5, width=0.6)
    ax2.plot(xs, vdf["历史累计毛利"].tolist(), color=C_RED, lw=2, marker="o", ms=3)
    _label_line(ax2, xs, vdf["历史累计毛利"].tolist(), fmt="{:,.0f}", size=7, color=C_RED, skip=4)
    _xticks(ax, vdf)
    ax.set_ylabel("历史累计真实消耗", fontproperties=_FP, fontsize=8)
    ax2.set_ylabel("历史累计毛利", fontproperties=_FP, fontsize=8)
    ax2.spines["top"].set_visible(False)
    ax.set_title("历史累计真实消耗 & 历史累计毛利", fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.legend([Patch(color=C_MUTED, alpha=0.5), Line2D([0], [0], color=C_RED, lw=2)],
              ["历史累计真实消耗", "历史累计毛利"], prop=_FP, fontsize=8, loc="upper left", framealpha=0.7)
    fig.tight_layout(); return _to_img(fig)


def chart_arppu(vdf):
    fig, ax = _fig(16, 4.5); xs = _x(vdf)
    ax.fill_between(xs, vdf["人均充值金额"].tolist(), alpha=0.10, color=C_GREEN)
    ax.plot(xs, vdf["人均充值金额"].tolist(), color=C_GREEN, lw=1.8, marker="o", ms=3, label="全量ARPPU")
    _label_line(ax, xs, vdf["人均充值金额"].tolist(), fmt="${:.2f}", size=7, color=C_GREEN, skip=4)
    ax.plot(xs, vdf["老用户ARPPU"].tolist(), color=C_DARK, lw=1.8, ls="--", marker="s", ms=3, label="老用户ARPPU")
    _label_line(ax, xs, vdf["老用户ARPPU"].tolist(), fmt="${:.2f}", size=7, color=C_DARK, offset=10, skip=4)
    _xticks(ax, vdf)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.1f"))
    ax.set_title("老用户ARPPU vs 全量ARPPU", fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.legend(prop=_FP, fontsize=8, framealpha=0.7, loc="upper left")
    fig.tight_layout(); return _to_img(fig)


def chart_payrate(vdf):
    fig, ax = _fig(16, 5); xs = _x(vdf)
    ax.plot(xs, vdf["新充付费率"].tolist(), color=C_BLUE, lw=1.8, marker="o",  ms=3, label="新充付费率%（新充/注册）")
    ax.plot(xs, vdf["老充付费率"].tolist(), color=C_GREEN, lw=1.8, ls="--", marker="s", ms=3, label="老充付费率%")
    ax.plot(xs, vdf["首充转化率"].tolist(), color=C_GOLD, lw=1.8, ls=":", marker="^", ms=3, label="首充转化率%")
    _xticks(ax, vdf)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
    ax.set_title("新充付费率 vs 老充付费率 & 首充转化率", fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.legend(prop=_FP, fontsize=8, framealpha=0.7, loc="upper left")
    fig.tight_layout(); return _to_img(fig)


def chart_users(vdf):
    fig, ax = _fig(16, 4.5); xs = _x(vdf); w = 0.35
    b1 = ax.bar([x - w/2 for x in xs], vdf["注册人数"].tolist(), width=w, color=C_BLUE,  alpha=0.65, label="注册人数")
    b2 = ax.bar([x + w/2 for x in xs], vdf["首充人数"].tolist(), width=w, color=C_GOLD, alpha=0.85, label="首充人数")
    _label_bars(ax, b1, fmt="{:,.0f}", size=7, offset=3)
    _label_bars(ax, b2, fmt="{:,.0f}", size=7, offset=3)
    _xticks(ax, vdf)
    ax.set_title("首充人数 & 注册人数", fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.legend(prop=_FP, fontsize=8, framealpha=0.7, loc="upper left")
    fig.tight_layout(); return _to_img(fig)


def chart_cac_bonus(vdf):
    """一级首充获客成本 & 总赠送充值比（基于有效日期）"""
    fig, ax = _fig(16, 4.5); ax2 = ax.twinx(); xs = _x(vdf)

    if CAC_COL in vdf.columns:
        cac = [float(r[CAC_COL]) if pd.notna(r.get(CAC_COL)) else None
               for _, r in vdf.iterrows()]
    else:
        log.warning(f"列「{CAC_COL}」不存在，获客成本图将留空")
        cac = [None] * len(vdf)

    bonus = vdf["总赠送充值比"].tolist()
    ax.plot(xs, cac, color=C_DARK, lw=1.8, marker="o", ms=4, label="获客成本$", zorder=3)
    _label_line(ax, xs, cac, fmt="${:.1f}", size=7, color=C_DARK, skip=4)
    bars = ax2.bar(xs, bonus, color=C_RED, alpha=0.4, width=0.55, label="赠送充值比%")
    _label_bars(ax2, bars, fmt="{:.1f}%", size=7, color=C_RED)
    _xticks(ax, vdf)
    ax.set_ylabel("获客成本$", fontproperties=_FP, fontsize=8)
    ax2.set_ylabel("赠送充值比%", fontproperties=_FP, fontsize=8)
    ax2.spines["top"].set_visible(False)
    ax.set_title("一级首充获客成本 & 总赠送充值比（获客成本已排除最新一天未完整数据）",
                 fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.legend([Line2D([0], [0], color=C_DARK, lw=2, marker='o'), Patch(color=C_RED, alpha=0.4)],
              ["获客成本$", "赠送充值比%"], prop=_FP, fontsize=8, loc="upper left", framealpha=0.7)
    fig.tight_layout(); return _to_img(fig)


def chart_bet_game(vdf):
    fig, ax = _fig(16, 4.5); ax2 = ax.twinx(); xs = _x(vdf)
    bet  = (vdf["投注金额"] / 1000).tolist()
    game = vdf["游戏盈利率"].tolist()
    bars = ax.bar(xs, bet, color=C_MUTED, alpha=0.55, width=0.6)
    _label_bars(ax, bars, fmt="{:.0f}K", size=7, offset=5)
    ax2.plot(xs, game, color=C_RED, lw=1.8, marker="D", ms=3)
    _label_line(ax2, xs, game, fmt="{:.1f}%", size=7, color=C_RED, skip=4)
    _xticks(ax, vdf)
    ax.set_ylabel("投注额(K)", fontproperties=_FP, fontsize=8)
    ax2.set_ylabel("游戏盈利率%", fontproperties=_FP, fontsize=8)
    ax2.spines["top"].set_visible(False)
    ax.set_title("日投注金额（千USD）& 游戏盈利率", fontproperties=_FP, fontsize=10, color=C_DARK)
    ax.legend([Patch(color=C_MUTED, alpha=0.55), Line2D([0], [0], color=C_RED, lw=2, marker='D')],
              ["投注额(K)", "游戏盈利率%"], prop=_FP, fontsize=8, loc="upper left", framealpha=0.7)
    fig.tight_layout(); return _to_img(fig)


def chart_repay_trend(df):
    """
    留存趋势图：★  改用 get_complete_series（已内部做有效日期坐标映射）。
    x 轴刻度基于有效日期的连续索引，标签用原始日期标签。
    """
    SERIES = [
        ("首充2日复充率",  "2日复充",  21.0, C_BLUE,   "-",   "o",  1),
        ("首充3日复充率",  "3日复充",  15.0, C_GREEN,  "--",  "s",  2),
        ("首充7日复充率",  "7日复充",  11.0, C_GOLD,   ":",   "^",  6),
        ("首充14日复充率", "14日复充",  8.0, C_RED,    "-.",  "D", 13),
        ("首充30日复充率", "30日复充",  6.0, C_ORANGE, "--",  "P", 29),
    ]
    fig, ax = plt.subplots(figsize=(17, 6), facecolor="white")
    ax.set_facecolor("white")
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#D8CEBD")
    ax.tick_params(colors=C_MUTED, labelsize=7)

    # 有效日期总数（用于 x 轴范围）
    vdf = valid_df(df)
    n_valid = len(vdf)

    for col, lbl, target, clr, ls_, mk, lag in SERIES:
        if col not in df.columns: continue
        xs_ok, vals_ok, dates_ok = get_complete_series(df, col, lag)
        if not xs_ok: continue
        ax.plot(xs_ok, vals_ok, color=clr, lw=1.8, ls=ls_, marker=mk, ms=3.5,
                label=f"{lbl}（目标{target:.0f}%）", zorder=3)
        miss_xs   = [x for x, v in zip(xs_ok, vals_ok) if v < target]
        miss_vals = [v for v in vals_ok if v < target]
        if miss_xs:
            ax.scatter(miss_xs, miss_vals, marker='x', color=C_RED,
                       s=55, zorder=6, linewidths=1.8)
        ax.plot([xs_ok[0], xs_ok[-1]], [target, target],
                color=clr, lw=0.9, ls=":", alpha=0.45, zorder=2)
        ax.annotate(f"目标{target:.0f}%",
                    xy=(xs_ok[-1], target), xytext=(4, 0),
                    textcoords="offset points",
                    ha="left", va="center", fontsize=7.5,
                    fontproperties=_FP, color=clr, alpha=0.9,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white",
                              ec=clr, alpha=0.85, lw=0.6))

    # x 轴：仅有效日期，自适应步长
    all_valid_xs = list(range(n_valid))
    all_valid_lb = vdf["日期标签"].tolist()
    step = max(1, n_valid // 20)
    ax.set_xticks(all_valid_xs[::step])
    ax.set_xticklabels(all_valid_lb[::step], fontproperties=_FP, fontsize=7)
    ax.set_xlim(-0.5, n_valid - 0.5)

    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_ylim(bottom=0)
    ax.set_title("首充复充率趋势（2/3/7/14/30日）— 仅完整观测数据，红×=不达标",
                 fontproperties=_FP, fontsize=10, color=C_DARK)

    hs, ls2 = ax.get_legend_handles_labels()
    hs.append(Line2D([0], [0], marker='x', color=C_RED, linestyle='none',
                     markersize=8, markeredgewidth=1.8, label="不达标"))
    ls2.append("不达标")
    ax.legend(hs, ls2, prop=_FP, fontsize=8.5, framealpha=0.92,
              loc="upper left",
              bbox_to_anchor=(1.01, 1.0),
              borderaxespad=0,
              edgecolor="#D8CEBD")
    fig.subplots_adjust(right=0.82)
    return _to_img(fig)


# ══════════════════════════════════════════════════════════════════════════════
# 5. PDF 构建
# ══════════════════════════════════════════════════════════════════════════════
W, H   = A4
MARGIN = 14 * mm
IW     = W - 2 * MARGIN


def build_pdf(df: pd.DataFrame, cfg: dict, pdf_path: Path):
    # ★ ：绘图用过滤后的有效数据集
    vdf = valid_df(df)

    doc = SimpleDocTemplate(str(pdf_path), pagesize=A4,
        leftMargin=MARGIN, rightMargin=MARGIN,
        topMargin=14 * mm, bottomMargin=12 * mm,
        title=f"{cfg['name']} 大盘汇总表")

    S = getSampleStyleSheet()
    def ps(name, **kw):
        kw.setdefault("fontName", "NotoSC")
        return ParagraphStyle(name, parent=S["Normal"], **kw)

    s_title = ps("t",  fontSize=16, textColor=colors.HexColor(C_DARK), fontName="NotoSC-B", leading=20, spaceAfter=2)
    s_sub   = ps("sb", fontSize=9,  textColor=colors.HexColor(C_MUTED), spaceAfter=4)
    s_sec   = ps("sc", fontSize=11, textColor=colors.HexColor(C_DARK),  fontName="NotoSC-B", spaceBefore=8, spaceAfter=4, leftIndent=8)
    s_note  = ps("n",  fontSize=7.5, textColor=colors.HexColor(C_MUTED), spaceBefore=2, spaceAfter=2, leading=11)
    s_lbl   = ps("l",  fontSize=7.5, textColor=colors.HexColor(C_MUTED), leading=10)
    s_val   = ps("v",  fontSize=13, textColor=colors.HexColor(C_DARK),  fontName="NotoSC-B", leading=15)

    TBL = TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0),  colors.HexColor("#F5F0E8")),
        ("BACKGROUND", (0, 1), (-1, -1), colors.white),
        ("BOX",        (0, 0), (-1, -1), 0.5, colors.HexColor("#E2D9CC")),
        ("INNERGRID",  (0, 0), (-1, -1), 0.3, colors.HexColor("#EDE7DB")),
        ("TOPPADDING",    (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING",   (0, 0), (-1, -1), 7),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 7),
        ("VALIGN",     (0, 0), (-1, -1), "MIDDLE"),
    ])

    def kpi3(items, cw=None):
        nc = len(items); cw = cw or [IW / nc] * nc
        td = [[Paragraph(a, s_lbl) for a, _, _ in items],
              [Paragraph(str(b), s_val) for _, b, _ in items],
              [Paragraph(str(c) if c else "", s_note) for _, _, c in items]]
        t = Table(td, colWidths=cw, hAlign="LEFT"); t.setStyle(TBL); return t

    def sec(txt):
        return Paragraph(f'<font color="{C_GOLD}">▌</font> {txt}', s_sec)

    # ── 数据 ──────────────────────────────────────────────────
    n    = len(df)
    last = df.iloc[-1]
    # today / yest 从原始 df 的有效行取（保持原逻辑）
    valid_dep = df[df["充值金额"].fillna(0) > 0]
    today = valid_dep.iloc[-1] if len(valid_dep) >= 1 else last
    yest  = valid_dep.iloc[-2] if len(valid_dep) >= 2 else today

    # 日充提差ROI：D-1 vs D-2
    roi_today = np.nan
    roi_yest  = np.nan
    if len(df) >= 3:
        roi_today = df["日充提差ROI"].iloc[-2]
        roi_yest  = df["日充提差ROI"].iloc[-3]
    elif len(df) >= 2:
        roi_today = df["日充提差ROI"].iloc[-2]

    # ★  新增：实时次留（首充2日复充率）D-1 vs D-2
    r2d_today = np.nan
    r2d_yest  = np.nan
    if "首充2日复充率" in df.columns:
        if len(df) >= 3:
            r2d_today = df["首充2日复充率"].iloc[-2]   # 昨日
            r2d_yest  = df["首充2日复充率"].iloc[-3]   # 前天
        elif len(df) >= 2:
            r2d_today = df["首充2日复充率"].iloc[-2]

    # 累计字段
    hist_consume  = last["历史累计真实消耗"]
    hist_profit   = last["历史累计毛利"]
    cum_roi_str   = f"{hist_profit / hist_consume * 100:.1f}%" if hist_consume > 0 else "N/A"
    cum_fdc_total = int(df["首充人数"].fillna(0).sum())

    active = df[df["充值金额"].fillna(0) > 0]
    n_active = len(active)

    def avgc(col):
        if col not in active.columns: return np.nan
        return active[col].dropna().mean()

    avg_consume = avgc("真实消耗")
    avg_deposit = avgc("充值金额")
    avg_fdc     = avgc("首充人数")
    avg_reg     = avgc("注册人数")
    avg_cac     = active[CAC_COL].dropna().mean() if CAC_COL in active.columns else np.nan

    r2_avg, r2_d0, r2_d1, _ = weighted_avg_7d(df, "首充2日复充率", 1)
    r2_str = f"{r2_avg:.1f}%" if not np.isnan(r2_avg) else "N/A"

    date_range = f"{df['日期标签'].iloc[0]} — {df['日期标签'].iloc[-1]}"
    now_str    = datetime.now().strftime("%Y.%m.%d %H:%M")

    CLR_UP = "#1A6636"; CLR_DN = "#8B1A14"

    def delta_p(col, inv=False):
        tv = today.get(col, np.nan); yv = yest.get(col, np.nan)
        if pd.isna(tv) or pd.isna(yv) or yv == 0: return Paragraph("─", s_note)
        d = tv - yv; pct = d / abs(yv) * 100
        arr = "▲" if d > 0 else "▼"; good = (d > 0) if not inv else (d < 0)
        c = CLR_UP if good else CLR_DN
        return Paragraph(f'<font color="{c}"><b>{arr}{abs(pct):.1f}%</b></font>', s_note)

    def yv_s(col, unit="", dec=1):
        v = yest.get(col, np.nan)
        if pd.isna(v): return "昨日 N/A"
        return f"昨日 {v:,.0f}{unit}" if dec == 0 else f"昨日 {v:.{dec}f}{unit}"

    def td_i(label, col, unit="", dec=1, inv=False):
        tv = today.get(col, np.nan)
        val = "N/A" if pd.isna(tv) else (f"{tv:,.0f}{unit}" if dec == 0 else f"{tv:.{dec}f}{unit}")
        return (label, val, yv_s(col, unit, dec), delta_p(col, inv))

    def td_roi():
        if pd.isna(roi_today):
            return ("日充提差ROI（昨日VS前天）", "N/A", "前天 N/A", Paragraph("─", s_note))
        value = f"{roi_today:.1f}%"
        if pd.isna(roi_yest) or roi_yest == 0:
            return ("日充提差ROI（昨日VS前天）", value, "前天 N/A", Paragraph("─", s_note))
        diff = roi_today - roi_yest; pct = diff / abs(roi_yest) * 100
        arrow = "▲" if diff > 0 else "▼"; color = CLR_UP if diff > 0 else CLR_DN
        delta = Paragraph(f'<font color="{color}"><b>{arrow}{abs(pct):.1f}%</b></font>', s_note)
        return ("日充提差ROI（昨日VS前天）", value, f"前天 {roi_yest:.1f}%", delta)

    # ★  新增：实时次留 KPI（D-1 vs D-2，因当日窗口未关闭）
    def td_r2d():
        if pd.isna(r2d_today):
            return ("实时次留（昨日VS前天）", "N/A", "前天 N/A", Paragraph("─", s_note))
        value = f"{r2d_today:.1f}%"
        if pd.isna(r2d_yest) or r2d_yest == 0:
            return ("实时次留（昨日VS前天）", value, "前天 N/A", Paragraph("─", s_note))
        diff = r2d_today - r2d_yest; pct = diff / abs(r2d_yest) * 100
        arrow = "▲" if diff > 0 else "▼"
        # 次留高好
        color = CLR_UP if diff > 0 else CLR_DN
        delta = Paragraph(f'<font color="{color}"><b>{arrow}{abs(pct):.1f}%</b></font>', s_note)
        return ("实时次留（昨日VS前天）", value, f"前天 {r2d_yest:.1f}%", delta)

    # ★  新增：裂变率 KPI（今日 vs 昨日）
    def td_fission():
        tv = today.get("裂变率", np.nan)
        yv = yest.get("裂变率", np.nan)
        val = "N/A" if pd.isna(tv) else f"{tv:.2f}%"
        yv_str = "昨日 N/A" if pd.isna(yv) else f"昨日 {yv:.2f}%"
        if pd.isna(tv) or pd.isna(yv) or yv == 0:
            return ("裂变率（非一级首充/充值人数）", val, yv_str, Paragraph("─", s_note))
        diff = tv - yv; pct = diff / abs(yv) * 100
        arrow = "▲" if diff > 0 else "▼"; color = CLR_UP if diff > 0 else CLR_DN
        delta = Paragraph(f'<font color="{color}"><b>{arrow}{abs(pct):.1f}%</b></font>', s_note)
        return ("裂变率（非一级首充/充值人数）", val, yv_str, delta)

    def kpi4(items, cw=None):
        nc = len(items); cw = cw or [IW / nc] * nc
        td = [[Paragraph(a, s_lbl) for a, _, _, _ in items],
              [Paragraph(str(b), s_val) for _, b, _, _ in items],
              [Paragraph(c, s_note) for _, _, c, _ in items],
              [d for _, _, _, d in items]]
        t = Table(td, colWidths=cw, hAlign="LEFT"); t.setStyle(TBL); return t

    def ret_summary_tbl():
        hdr    = ps("rh",  fontName="NotoSC-B", fontSize=8.5, textColor=colors.HexColor(C_DARK))
        cell   = ps("rc",  fontName="NotoSC",   fontSize=9,   textColor=colors.HexColor(C_DARK))
        note_c = ps("rnc", fontName="NotoSC",   fontSize=7.5, textColor=colors.HexColor(C_MUTED), leading=11)
        gn     = ps("rg",  fontName="NotoSC-B", fontSize=9,   textColor=colors.HexColor(C_GREEN))
        rd     = ps("rr",  fontName="NotoSC-B", fontSize=9,   textColor=colors.HexColor(C_RED))
        rows = [[Paragraph(t, hdr) for t in ["指标", "目标", "实际留存率（近7天）", "差距(pp)", "备注（数据区间）"]]]
        for label, col, target, lag in RETENTION_TARGETS:
            if col not in df.columns:
                rows.append([Paragraph(s, cell) for s in [label, f"{target:.0f}%", "N/A", "─", "─"]])
                continue
            avg, d0, d1, nd = weighted_avg_7d(df, col, lag)
            if nd == 0:
                remark = f"数据不足（行数{n} < lag{lag}+1）"
            else:
                remark = f"{d0}~{d1}（{nd}天）"
            if np.isnan(avg):
                rows.append([Paragraph(label, cell), Paragraph(f"{target:.0f}%", cell),
                             Paragraph("N/A", cell), Paragraph("─", cell), Paragraph(remark, note_c)])
            else:
                gap = avg - target; ok = gap >= 0; gs = gn if ok else rd
                rows.append([Paragraph(label, cell), Paragraph(f"{target:.0f}%", cell),
                             Paragraph(f"{avg:.1f}%", gs), Paragraph(f"{gap:+.1f}", gs),
                             Paragraph(remark, note_c)])
        cw = [IW * p for p in [0.18, 0.11, 0.22, 0.15, 0.34]]
        tbl = Table(rows, colWidths=cw, hAlign="LEFT")
        tbl.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#F5F0E8")),
            ("BOX",        (0, 0), (-1, -1), 0.5, colors.HexColor("#E2D9CC")),
            ("INNERGRID",  (0, 0), (-1, -1), 0.3, colors.HexColor("#EDE7DB")),
            ("TOPPADDING",    (0, 0), (-1, -1), 7),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
            ("LEFTPADDING",   (0, 0), (-1, -1), 8),
            ("RIGHTPADDING",  (0, 0), (-1, -1), 8),
            ("VALIGN",     (0, 0), (-1, -1), "MIDDLE"),
        ]))
        return tbl

    # ── Story ──────────────────────────────────────────────────
    story = []
    story.append(Paragraph(f"{cfg['name']} 大盘平台汇总表", s_title))
    story.append(Paragraph(
        f"数据区间：{date_range}（近{n}日）&nbsp;&nbsp;|&nbsp;&nbsp;生成：{now_str}", s_sub))
    story.append(HRFlowable(width=IW, thickness=1.5, color=colors.HexColor(C_GOLD), spaceAfter=6))

    # 累计汇总
    story.append(sec("累计汇总"))
    story.append(kpi3([
        ("历史累计真实消耗",   f"{hist_consume:,.0f}",  "取最新日累计字段"),
        ("历史累计毛利",       f"{hist_profit:,.0f}",   "取最新日累计字段"),
        ("累计毛利ROI",        cum_roi_str,              "历史累计毛利÷历史累计真实消耗"),
        ("累计首充人数",       f"{cum_fdc_total:,}",    "本期合计"),
        ("日均真实消耗",
         f"{avg_consume:,.0f}" if pd.notna(avg_consume) else "N/A",
         f"近{n_active}日均值（排除最新一天未完整）"),
    ]))
    story.append(Spacer(1, 4))
    story.append(kpi3([
        ("日均充值金额",           f"{avg_deposit:,.0f}" if pd.notna(avg_deposit) else "N/A", f"近{n_active}日有效均值"),
        ("日均首充人数",           f"{avg_fdc:,.0f}"     if pd.notna(avg_fdc)     else "N/A", f"近{n_active}日有效均值"),
        ("日均注册人数",           f"{avg_reg:,.0f}"     if pd.notna(avg_reg)     else "N/A", f"近{n_active}日有效均值"),
        ("一级首充获客成本（均）", f"${avg_cac:.2f}"     if pd.notna(avg_cac)     else "N/A", "排除最新一天未完整"),
        ("次留均值（近7天）",      r2_str,                                                     f"2日复充 {r2_d0}~{r2_d1}"),
    ]))
    story.append(Spacer(1, 6))

    # 今日 vs 昨日（：新增实时次留和裂变率，共三行）
    story.append(sec("今日 vs 昨日"))
    story.append(kpi4([
        td_i("今日充值金额", "充值金额", dec=0),
        td_i("充提差",       "充提差",   dec=0),
        td_i("充提差比",     "充提差比", unit="%"),
        td_i("首充人数",     "首充人数", dec=0),
        td_i("注册人数",     "注册人数", dec=0),
    ]))
    story.append(Spacer(1, 4))
    story.append(kpi4([
        td_i("全量ARPPU",   "人均充值金额", unit="$", dec=2),
        td_i("新充付费率",  "新充付费率",  unit="%"),
        td_i("老充付费率",  "老充付费率",  unit="%"),
        td_i("游戏盈利率",  "游戏盈利率",  unit="%"),
        td_roi(),
    ]))
    story.append(Spacer(1, 4))
    # ★  新增一行：实时次留 + 裂变率（仅两格，各占二分之一宽）
    story.append(kpi4(
        [td_r2d(), td_fission()],
        cw=[IW / 5, IW / 5],
    ))
    story.append(Spacer(1, 6))

    # 留存趋势图（★ ：用 KeepTogether 确保标题/说明/图表在同一页）
    story.append(KeepTogether([
        sec("首充复充率趋势（2/3/7/14/30日）"),
        Paragraph(
            "仅显示观测窗口已关闭的完整数据（各指标截止日期不同）。"
            "目标：2日21% / 3日15% / 7日11% / 14日8% / 30日6%。红色 × = 当日不达标。",
            s_note),
        Spacer(1, 3),
        Image(chart_repay_trend(df), width=IW, height=IW * 6 / 17),
        Spacer(1, 8),
    ]))

    # 留存目标汇总表
    story.append(KeepTogether([
        sec("首充留存/复充目标完成情况（近7天）"),
        Spacer(1, 3),
        Paragraph(
            "实际留存率 = 近7天完整数据加权均值（∑各日复充率×首充人数 ÷ ∑首充人数）。"
            "各指标观测窗口不同，具体日期见备注列。差距 = 实际 − 目标（pp=百分点）。",
            s_note),
        Spacer(1, 3),
        ret_summary_tbl(),
    ]))
    story.append(Spacer(1, 10))

    def add_chart(buf, title, h=4.5):
        story.append(sec(title))
        story.append(Image(buf, width=IW, height=IW * h / 16))
        story.append(Spacer(1, 6))

    log.info(f"[{cfg['name']}] 生成常规图表...")
    # ★ ：全部传入 vdf（有效数据集）
    add_chart(chart_dep_cost(vdf),   "日充值金额 vs 日真实消耗")
    add_chart(chart_diff(vdf),       "充提差 & 充提差比")
    add_chart(chart_roi(vdf),        "充提差比 vs 累计毛利ROI", h=5)
    add_chart(chart_cum_profit(vdf), "历史累计真实消耗 & 历史累计毛利", h=5)
    add_chart(chart_arppu(vdf),      "老用户ARPPU vs 全量ARPPU")
    add_chart(chart_payrate(vdf),    "新充付费率 vs 老充付费率 & 首充转化率", h=5)
    add_chart(chart_users(vdf),      "首充人数 & 注册人数")
    add_chart(chart_cac_bonus(vdf),  "一级首充获客成本 & 总赠送充值比")
    add_chart(chart_bet_game(vdf),   "日投注金额（千USD）& 游戏盈利率")

    story.append(HRFlowable(width=IW, thickness=0.5,
                             color=colors.HexColor("#E2D9CC"), spaceBefore=4))
    fn = ps("fn", fontSize=7.5, textColor=colors.HexColor(C_MUTED), leading=11)
    note_text = (
        "① 真实消耗/一级首充获客成本排除最新一天（当日录入未完整）；<br/>"
        "② 留存汇总表取近7天完整数据加权均值，随每日数据自动更新，日期见备注列。<br/>"
        "③ 日充提差ROI=充提差/真实消耗，因为当天真实消耗未录完，使用昨日VS前天。<br/>"
        "④ 实时次留=首充2日复充率，因当日窗口次留未完整，取昨日VS前天进行比较。<br/>"
        "⑤ 裂变率=非一级首充人数/充值人数。<br/>"
    )
    story.append(Paragraph(note_text, fn))

    log.info(f"[{cfg['name']}] 构建 PDF ...")
    doc.build(story)
    log.info(f"[{cfg['name']}] ✅ PDF 保存：{pdf_path}")


# ══════════════════════════════════════════════════════════════════════════════
#  Telegram 推送
# ══════════════════════════════════════════════════════════════════════════════

def send_pdf(pdf_path: Path, cfg: dict) -> bool:
    caption = (f"📊 *{cfg['name']} 大盘汇总表*\n"
               f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    try:
        with open(pdf_path, "rb") as f:
            r = requests.post(
                f"https://api.telegram.org/bot{cfg['bot_token']}/sendDocument",
                data={"chat_id": cfg["chat_id"], "caption": caption,
                      "parse_mode": "Markdown"},
                files={"document": (pdf_path.name, f, "application/pdf")},
                timeout=120)
        ok = r.json().get("ok", False)
        log.info(f"[{cfg['name']}] TG " + ("✅ 成功" if ok else f"❌ {r.json().get('description')}"))
        return ok
    except Exception as e:
        log.error(f"[{cfg['name']}] ❌ 发送异常：{e}")
        return False


# ══════════════════════════════════════════════════════════════════════════════
# 主流程
# ══════════════════════════════════════════════════════════════════════════════

def run_platform(cfg: dict):
    log.info(f"\n{'=' * 55}")
    log.info(f"  平台：{cfg['name']}")
    log.info(f"{'=' * 55}")
    try:
        path = find_excel(cfg)
        df   = load_df(path)
        log.info(f"[{cfg['name']}] {df['日期标签'].iloc[0]} → {df['日期标签'].iloc[-1]}（{len(df)}日）")
        Path(cfg["output_dir"]).mkdir(parents=True, exist_ok=True)
        latest_date = str(df["日期"].iloc[-1])[:8]
        pdf_path = Path(cfg["output_dir"]) / f"{cfg['name']}_大盘汇总表_{latest_date}.pdf"
        build_pdf(df, cfg, pdf_path)
        send_pdf(pdf_path, cfg)
    except Exception as e:
        log.error(f"[{cfg['name']}] ❌ 失败：{e}", exc_info=True)


def main():
    log.info(f"\n{'=' * 55}")
    log.info(f"  多平台大盘日报   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log.info(f"{'=' * 55}")
    for cfg in PLATFORMS:
        run_platform(cfg)
    log.info("\n✅ 全部完成")


if __name__ == "__main__":
    main()

09:45:40  
09:45:40    多平台大盘日报   2026-06-20 09:45:40
09:45:40  =======================================================
09:45:40  
09:45:40    平台：Afun_mx
09:45:40  =======================================================
09:45:40  [Afun_mx] 读取：日报-大盘日报_20260619.xlsx
09:45:41  [load_df] Excel 列名列表：['日期', '历史累计毛利', '历史累计充提差', '历史累计真实消耗', '历史累计充提ROI', '真实消耗', '一级首充获客成本', '首充LTV7日', '首充LTV14日', '首充LTV30日', '充提差', '充提差比', '充值金额', '充值人数', '人均充值金额', '人均充值笔数', '首充人数', '新充人数', '首充人均充值笔数', '首充人均充值金额', '一级首充人数', '一级新充人数', '非一级充值人数', '非一级首充人数/充值人数', '非一级充值人数/充值人数', '非一级首充人数/首充人数', '平台首充用户赢钱比例', '平台非首充用户赢钱比例', '注册人数', '新充付费率', '首充转化率', '首充2日复登率', '首充3日复登率', '首充7日复登率', '首充14日复登率', '首充30日复登率', '首充2日复充率', '首充3日复充率', '首充7日复充率', '首充14日复充率', '首充30日复充率', '首充2日复投率', '首充3日复投率', '首充7日复投率', '首充30日复投率', '老充付费率', '老用户充值金额', '出入比', '游戏盈利率', '投注金额', '投充比', '投注人数', '充投人数', '获得佣金人数', '获得佣金金额', '总赠送充值比', '彩金赠送充值比', '非一级首充人数', '首充投注赢钱人数']
09:45:41  [Afun_mx] 04/20 → 06/18（60日）
09:45:41  [Afun_mx] 生成常规图表...
09:45:54  [Af